# Black Summer 2019–20 — Entity Liability

End-to-end liability with the **corrected apportionment convention**: each entity is charged
its share of **total** anthropogenic warming (`global_share`), not its share of the Carbon
Majors subtotal. The Carbon Majors collectively absorb ~75% of climate-attributed damages;
the rest is attributable to emitters outside the database.

- **PR/FAR**: nonstationary GEV shift-fit, primary PR ≈ 4.0, FAR ≈ 0.75 (notebook 04).
- **Uncertainty**: from the PR bootstrap (FaIR ensemble cancels in every warming *share*).
- **Damage scenarios** are the discrete damage axis; the same primary FAR is applied to each.
  A separate PR × damages grid shows the joint sensitivity.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('../../').resolve()
sys.path.insert(0, str(ROOT))
from src.attribution import (
    area_weighted_series, season_block_max, wet_season_max_ndays,
    load_gmst, extrapolate_to, smoothed_covariate, event_gmst_sigma,
    shift_fit_gev, fit_gev, build_liability_table, far,
    AUD_TO_USD, CC_RATE_STANDARD, CC_RATE_HIGH, CLIM_START, CLIM_END,
)
from scipy.stats import genextreme

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
FIGS = ROOT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)
print('Setup complete.')


In [ ]:
ew = pd.read_parquet(PROC / 'entity_warming_contribution.parquet')
print(f'Carbon Majors coverage of global fossil CO2: {ew["global_share"].sum()*100:.1f}%')

# PR from notebook 04 (primary) + bootstrap samples for FAR uncertainty
pr_df = pd.read_csv(PROC / 'black_summer_pr_era5.csv')
primary_row = pr_df[pr_df['method'].str.startswith('primary')].iloc[0]
PR_PRIMARY = float(primary_row['pr'])
boot = pd.read_parquet(PROC / 'black_summer_pr_shiftfit_bootstrap.parquet')['pr_boot'].values
print(f'Primary PR = {PR_PRIMARY:.2f}, FAR = {far(PR_PRIMARY):.3f}  '
      f'(bootstrap n={len(boot)})')

FX = AUD_TO_USD[2020]
scenarios = {
    'conservative':  dict(damages_usd_b=2.32 * FX,  pr=PR_PRIMARY, pr_samples=boot,
                          label='Insured losses (ICA), AUD 2.32B'),
    'central':       dict(damages_usd_b=10.0 * FX,  pr=PR_PRIMARY, pr_samples=boot,
                          label='Direct economic, AUD 10B'),
    'comprehensive': dict(damages_usd_b=103.0 * FX, pr=PR_PRIMARY, pr_samples=boot,
                          label='Total social cost, AUD 103B'),
}


In [ ]:
liability, totals = build_liability_table(ew, scenarios)
liability.to_parquet(PROC / 'black_summer_liability.parquet', index=False)
totals.to_csv(PROC / 'black_summer_scenario_totals.csv', index=False)

print('Scenario totals (Carbon Majors, ~75% of global):')
print(totals[['scenario', 'damages_usd_b', 'far', 'far_p05', 'far_p95',
              'total_attributed_usd_b']].to_string(index=False))
print(f'\nPRIMARY (central): USD {totals.set_index("scenario").loc["central","total_attributed_usd_b"]:.2f}B '
      f'across Carbon Majors')
print('\nTop 10 entities — central scenario (USD M, with 5–95% PR uncertainty):')
cols = ['rank', 'parent_entity', 'parent_type', 'liability_central_USD_M',
        'liability_central_p05_USD_M', 'liability_central_p95_USD_M']
top10 = liability.head(10)[cols].copy()
for c in cols[3:]:
    top10[c] = top10[c].map('{:,.1f}'.format)
print(top10.to_string(index=False))


In [ ]:
# ── Figure: top-20 entities, central scenario with PR-uncertainty bars ──
top20 = liability.head(20)
type_colors = {'Investor-owned Company': '#2196F3', 'State-owned Entity': '#FF5722',
               'Nation State': '#4CAF50'}
colors = top20['parent_type'].map(type_colors).fillna('#9E9E9E')
fig, ax = plt.subplots(figsize=(11, 8))
y = np.arange(len(top20))
ax.barh(y, top20['liability_central_USD_M'], color=colors, alpha=0.85)
xerr_lo = (top20['liability_central_USD_M'] - top20['liability_central_p05_USD_M']).clip(lower=0)
xerr_hi = (top20['liability_central_p95_USD_M'] - top20['liability_central_USD_M']).clip(lower=0)
ax.errorbar(top20['liability_central_USD_M'], y, xerr=[xerr_lo, xerr_hi],
            fmt='none', color='#333', lw=1, capsize=3, alpha=0.6, label='PR 5–95% (bootstrap)')
ax.set_yticks(y); ax.set_yticklabels(top20['parent_entity'], fontsize=9); ax.invert_yaxis()
ax.set_xlabel('Attributed liability — central scenario (USD millions)')
ax.set_title('Black Summer 2019–20: entity liability\n'
             'global warming share × FAR(0.75) × AUD 10B damages', fontsize=12)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=c, label=t) for t, c in type_colors.items()] +
                  [Patch(facecolor='#333', label='PR 5–95% (bootstrap)')],
          fontsize=8, loc='lower right')
plt.tight_layout(); plt.savefig(FIGS / 'black_summer_liability_top20.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Joint PR × damages sensitivity (Saudi Aramco) ──
aramco = float(liability.loc[liability['parent_entity'] == 'Saudi Aramco', 'global_share'].iloc[0])
pr_range = [2, 3, 4, 6, 9, 12]
dmg_aud  = [2.32, 5, 10, 25, 50, 103]
grid = pd.DataFrame(
    index=[f'PR={p}' for p in pr_range],
    columns=[f'AUD {d}B' for d in dmg_aud],
    data=[[aramco * far(p) * d * FX * 1000 for d in dmg_aud] for p in pr_range])
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(grid.astype(float), annot=True, fmt='.0f', cmap='YlOrRd',
            cbar_kws={'label': 'USD millions'}, linewidths=0.5, ax=ax)
ax.set_title('Saudi Aramco — Black Summer liability sensitivity (USD M)\n'
             'global-share apportionment', fontsize=11)
ax.set_xlabel('Total damages'); ax.set_ylabel('Probability Ratio')
plt.tight_layout(); plt.savefig(FIGS / 'black_summer_sensitivity_aramco.png', bbox_inches='tight')
plt.show()
print(f'Saudi Aramco global warming share: {aramco*100:.3f}%')
print(f'Saudi Aramco central liability: USD '
      f'{liability.loc[liability.parent_entity=="Saudi Aramco","liability_central_USD_M"].iloc[0]:.0f}M')
